# LoRA, QLoRA & Quantization in SageMaker

**Track:** Applied Agent Engineering Foundations  
**Module fit:** M1 — LLM Foundations  
**Runtime target:** SageMaker notebook on a GPU instance preferred; CPU fallback works for the LoRA-only path with a tiny model.

## Why this notebook exists
You uploaded a LoRA/QLoRA notebook. This classroom version keeps the same intent, but makes it:
- SageMaker friendly
- Azure free
- S3-input aware
- local-output only
- easier for learners to follow cell by cell

## What learners will understand
1. Why full fine-tuning is expensive
2. What LoRA changes inside a model
3. What QLoRA adds through low-bit loading
4. How trainable-parameter count drops sharply
5. How to run a small classroom-safe adaptation experiment

## Classroom rule
Use **tiny data + tiny steps** in class. The goal is understanding the workflow, not training a perfect model.


## Instructor talking points

- **Full fine-tuning** updates all parameters.
- **LoRA** freezes the base model and trains small low-rank adapters.
- **QLoRA** keeps the base model in quantized form and trains adapters on top.
- In a real enterprise setting, these methods are useful when you want domain adaptation without the full cost of end-to-end fine-tuning.
- In class, we keep outputs local inside the notebook session. We do **not** write back to S3.


In [ ]:

# Uncomment only if the notebook kernel is missing these packages.
# %pip install -q transformers datasets peft accelerate sentencepiece bitsandbytes pandas boto3 matplotlib


## Step 1 — Configuration

Update the bucket/key only if your training text already lives in S3.

**Accepted S3 formats for this classroom notebook**
- `.csv` with a `text` column
- `.jsonl` with a `text` field
- plain `.txt`

If `USE_S3_DATA = False`, the notebook uses a tiny built-in dataset so the flow always remains teachable.


In [ ]:

from __future__ import annotations

import io
import json
import math
import os
from pathlib import Path

import boto3
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

# --- Data settings ---
USE_S3_DATA = False
S3_BUCKET = "replace-me"
S3_KEY = "replace-me/training_text.csv"

# --- Model settings ---
BASE_MODEL_ID = "sshleifer/tiny-gpt2"   # classroom-safe default
# For a bigger GPU-backed demo you can switch to: "gpt2"
LOAD_IN_4BIT_IF_POSSIBLE = True

# --- LoRA settings ---
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["c_attn"]   # valid for GPT-2 family

# --- Training settings ---
MAX_SAMPLES = 64
MAX_LENGTH = 128
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 2
TRAIN_STEPS = 10

# --- Local artifact path (no S3 writes) ---
WORK_DIR = Path("./m1_lora_lab_artifacts")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Using region:", AWS_REGION)
print("Artifacts will stay local in:", WORK_DIR.resolve())


## Step 2 — Load a tiny dataset

This cell supports **S3 read-only input**. That matches your delivery constraint:
- documents can live in S3
- the notebook reads them
- any derived metrics stay local in DataFrames or local files


In [ ]:

s3 = boto3.client("s3", region_name=AWS_REGION)

def load_text_dataframe_from_s3(bucket: str, key: str) -> pd.DataFrame:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()

    if key.endswith(".csv"):
        return pd.read_csv(io.BytesIO(body))
    if key.endswith(".jsonl"):
        rows = [json.loads(line) for line in body.decode("utf-8").splitlines() if line.strip()]
        return pd.DataFrame(rows)
    if key.endswith(".txt"):
        lines = [x.strip() for x in body.decode("utf-8").splitlines() if x.strip()]
        return pd.DataFrame({"text": lines})

    raise ValueError("Supported formats: .csv, .jsonl, .txt")

if USE_S3_DATA:
    train_df = load_text_dataframe_from_s3(S3_BUCKET, S3_KEY)
else:
    train_df = pd.DataFrame({
        "text": [
            "Our HR assistant should answer only from approved policy context.",
            "Leave requests must include employee id, number of days, and reason.",
            "Enterprise agents should log tool calls for auditability.",
            "RAG systems improve groundedness by retrieving relevant context first.",
            "Error handling is critical in production AI pipelines.",
            "Prompt design affects tone, structure, and completeness of output.",
            "Observability makes debugging enterprise AI systems much easier.",
            "Bedrock models can be called from SageMaker notebooks using boto3.",
        ] * 8
    })

if "text" not in train_df.columns:
    raise ValueError("Training data must contain a 'text' column.")

train_df = train_df.dropna(subset=["text"]).head(MAX_SAMPLES).reset_index(drop=True)
display(train_df.head())
print("Rows loaded:", len(train_df))


## Step 3 — Tokenizer and model loading

This notebook chooses the path automatically:
- **QLoRA path** when CUDA + bitsandbytes are available
- **LoRA-only path** otherwise

That makes the notebook safe for mixed classroom environments.


In [ ]:

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_cuda = torch.cuda.is_available()
use_4bit = bool(use_cuda and LOAD_IN_4BIT_IF_POSSIBLE)

print("CUDA available:", use_cuda)
print("Attempt 4-bit load:", use_4bit)

model_load_notes = []

try:
    if use_4bit:
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            load_in_4bit=True,
            device_map="auto"
        )
        model = prepare_model_for_kbit_training(model)
        model_load_notes.append("Loaded base model in 4-bit mode (QLoRA path).")
    else:
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID)
        if use_cuda:
            model = model.to("cuda")
        model_load_notes.append("Loaded base model in regular precision (LoRA path).")
except Exception as e:
    print("4-bit or device-specific load failed. Falling back to standard loading.")
    print("Reason:", e)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID)
    if use_cuda:
        model = model.to("cuda")
    use_4bit = False
    model_load_notes.append("Fallback to regular precision loading.")

print("\n".join(model_load_notes))


## Step 4 — Apply LoRA adapters


In [ ]:

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def parameter_report(model) -> pd.DataFrame:
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params

    return pd.DataFrame([{
        "total_params": total_params,
        "trainable_params": trainable_params,
        "frozen_params": frozen_params,
        "pct_trainable": round(100 * trainable_params / total_params, 4),
        "mode": "QLoRA" if use_4bit else "LoRA",
        "base_model": BASE_MODEL_ID,
    }])

param_df = parameter_report(model)
display(param_df)


## Step 5 — Create a tokenized training dataset


In [ ]:

dataset = Dataset.from_pandas(train_df[["text"]])

def tokenize_batch(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )
    out["labels"] = out["input_ids"].copy()
    return out

tokenized_ds = dataset.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized_ds.set_format(type="torch")
print(tokenized_ds[0].keys())


## Step 6 — Minimal classroom training loop

This loop is intentionally short.
- It proves that adapter training is happening.
- It avoids turning the class into a long waiting session.


In [ ]:

from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
model.train()

loader = DataLoader(tokenized_ds, batch_size=BATCH_SIZE, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

loss_rows = []
global_step = 0

for epoch in range(10):
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()

        if (global_step + 1) % GRAD_ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()

        loss_rows.append({
            "step": global_step,
            "loss": float(loss.item() * GRAD_ACCUM_STEPS)
        })
        global_step += 1

        if global_step >= TRAIN_STEPS:
            break
    if global_step >= TRAIN_STEPS:
        break

loss_df = pd.DataFrame(loss_rows)
display(loss_df.head())
print("Completed steps:", len(loss_df))


## Step 7 — Visualize the short run


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(loss_df["step"], loss_df["loss"], marker="o")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("Short classroom LoRA/QLoRA run")
plt.grid(True)
plt.show()


## Step 8 — Compare generation before and after adaptation

For a proper scientific comparison you would save and evaluate a clean baseline first.
In class, we use a lightweight approximation:
- use the current adapted model
- compare outputs across prompts
- discuss whether the model sounds more aligned to the domain language


In [ ]:

model.eval()

def generate_text(prompt: str, max_new_tokens: int = 50) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

test_prompts = [
    "Enterprise AI systems need observability because",
    "A leave request workflow should capture",
    "RAG helps reduce hallucination by",
]

sample_outputs = pd.DataFrame({
    "prompt": test_prompts,
    "adapted_output": [generate_text(p, max_new_tokens=40) for p in test_prompts]
})

display(sample_outputs)


## Step 9 — Save locally

We keep artifacts local in the SageMaker notebook file system.
That is consistent with your delivery model: **read from S3, but do not write back to S3**.


In [ ]:

param_df.to_csv(WORK_DIR / "parameter_report.csv", index=False)
loss_df.to_csv(WORK_DIR / "loss_curve.csv", index=False)
sample_outputs.to_csv(WORK_DIR / "sample_generations.csv", index=False)

print("Saved:")
for p in sorted(WORK_DIR.glob("*")):
    print("-", p)


## Mini-hack — M1 adapter challenge

Ask learners to do **one** of these:

1. Change the training text so the model sounds like an HR assistant.
2. Change `LORA_R` and explain how trainable parameter count changes.
3. Switch from `sshleifer/tiny-gpt2` to `gpt2` on GPU and compare latency.
4. Run with and without the 4-bit path and explain the operational trade-off.

## Reflection questions
- What problem does LoRA solve?
- What extra benefit does QLoRA add?
- When would you still prefer full fine-tuning?
- What risks appear when using heavily quantized models?


## Instructor wrap-up

This notebook is deliberately small, but the workflow is real:
**load model → apply adapters → train a little → inspect cost/parameter savings → keep artifacts local**.

That is enough for learners to understand the adapter-based fine-tuning story before they move into RAG and agents.
